In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import sys
import uproot
import numpy as np
import math
import pandas as pd
import matplotlib.pyplot as plt
sys.path.append("../../")
import data_loading as dl
from importlib import reload
reload(dl)

from microfit import run_plotter as rp
from microfit import histogram as hist

from microfit import variable_definitions as vdef
from microfit import selections

In [3]:
RUN = ["3"]
#RUN = ["1","2","3"] #important that it's a string 1) new format to include latest runs 2) to include 'mc_pdg' otherwise it gets dropped

rundata, mc_weights, data_pot = dl.load_runs(
    RUN,
    data="bnb",
    loadpi0variables=False,
    loadshowervariables=True,
    loadrecoveryvars=False,
    loadsystematics=True,
    numupresel=False,
    loadnumuvariables=False,
    use_bdt=False,
    load_lee=False,
    load_nue_tki=True,
    blinded=True,
    load_crt_vars=False,
    enable_cache=True,
)


Loading run 3
/exp/uboone/data/users/cthorpe/PELEE_2023_Samples/run3/nuepresel/bnb_beam_off_peleeTuple_uboone_v08_00_00_70_run3.root


/exp/uboone/app/users/mmoudgal/miniforge3/envs/python3LEE/lib/python3.7/site-packages/pandas/core/series.py:679: RuntimeWarning: invalid value encountered in sqrt
  result = getattr(ufunc, method)(*inputs, **kwargs)


Calc true TKI variables for leading proton only
Calc reco TKI variables for leading proton only


/exp/uboone/app/users/mmoudgal/miniforge3/envs/python3LEE/lib/python3.7/site-packages/pandas/core/generic.py:2505: PerformanceWarning: 
your performance may suffer as PyTables will pickle object types that it cannot
map directly to c-types [inferred_type->mixed,key->block5_values] [items->Index(['mc_pdg', 'TrueProtonIdx', 'TrueElecIndices', 'Signal_1eNp',
       'Signal_1e1p', 'PFPStartsInPCV_v', 'IsContained_v',
       'RecoShowersIndices', 'RecoProtonIndices', 'Sel_1e1p'],
      dtype='object')]

  encoding=encoding,


/exp/uboone/data/users/cthorpe/PELEE_2023_Samples/run3/nuepresel/overlay_peleeTuple_uboone_v08_00_00_70_run3_nu.root


../../data_loading.py:801: RuntimeWarning: invalid value encountered in true_divide
  df["proton_pz"] = np.where((mc_E_prot > 0), mc_pz_prot / mc_p_prot, np.nan)


Calc true TKI variables for leading proton only
Calc reco TKI variables for leading proton only


/exp/uboone/app/users/mmoudgal/miniforge3/envs/python3LEE/lib/python3.7/site-packages/pandas/core/generic.py:2505: PerformanceWarning: 
your performance may suffer as PyTables will pickle object types that it cannot
map directly to c-types [inferred_type->mixed,key->block5_values] [items->Index(['weightsGenie', 'weightsFlux', 'weightsReint', 'mc_pdg',
       'TrueProtonIdx', 'TrueElecIndices', 'Signal_1eNp', 'Signal_1e1p',
       'PFPStartsInPCV_v', 'IsContained_v', 'RecoShowersIndices',
       'RecoProtonIndices', 'Sel_1e1p'],
      dtype='object')]

  encoding=encoding,


/exp/uboone/data/users/cthorpe/PELEE_2023_Samples/run3/nuepresel/overlay_peleeTuple_uboone_v08_00_00_70_run3_nue.root
Calc true TKI variables for leading proton only
Calc reco TKI variables for leading proton only
/exp/uboone/data/users/cthorpe/PELEE_2023_Samples/run3/nuepresel/overlay_peleeTuple_uboone_v08_00_00_70_run3_dirt.root
Calc true TKI variables for leading proton only
Calc reco TKI variables for leading proton only


In [5]:
print(rundata.keys())
#from particle import Particle
print("sel_1e1p_w_cuts" in rundata["mc"].columns)
print("mc_signal_1e1p" in rundata["mc"].columns)

dict_keys(['data', 'ext', 'mc', 'nue', 'drt'])
True
True


In [6]:
rundata["mc"].loc[:, ('category_1e1p','Signal_1e1p', 'mc_signal_1e1p','Sel_1e1p','sel_1e1p_w_cuts')].head(10)

,category_1e1p,Signal_1e1p,mc_signal_1e1p,Sel_1e1p,sel_1e1p_w_cuts
entry,,,,,
0,2,False,False,False,False
1,31,False,False,False,False
2,3,False,False,False,False
3,5,False,False,False,False
4,31,False,False,True,True
5,2,False,False,False,False
7,3,False,False,False,False
8,31,False,False,False,False
9,2,False,False,False,False


In [12]:
import numpy as np
import math
from numu_tki import tki_calculators

# Functions for setting the signal definition selection in reco variables and adding useful variables for the CC1e1P selection
# (original framework in Root/C++ developed by S Gardiner, re-written into the python PeLEE framework by C Thorpe)
# Author: M Moudgalya

################################################################################
# Fiducial volume cut is already applied by preselection filter - repeated here
# for completeness

# Definitions from SG's code
'''
FV_X_MIN = 21.5
FV_X_MAX = 234.85
FV_Y_MIN = -95.0
FV_Y_MAX = 95.0
FV_Z_MIN = 21.5
FV_Z_MAX = 966.8
DEAD_Z_MIN = 10000 # SG's code does not cut dead region 
DEAD_Z_MAX = 10000
'''

# Definitions in PeLEE technote and 'selected' variable
# https://github.com/ubneutrinos/searchingfornues/blob/0489ac5457335a553a3bab54ee5d7ba91734adf0/Selection/SelectionTools/CC0piNpSelection_tool.cc#L97-L102
# https://github.com/ubneutrinos/searchingfornues/blob/889002e5ec93b567265c3af8c178172363200490/Selection/SelectionTools/CC0piNpSelection_tool.cc#L415-L420
FV_X_MIN =   10.0
FV_X_MAX =  246.4
FV_Y_MIN = -101.5
FV_Y_MAX =  101.5
FV_Z_MIN =   10.0
FV_Z_MAX =  986.8
DEAD_Z_MIN = 675 # SG's code does not cut dead region 
DEAD_Z_MAX = 775

def sel_reco_vertex_in_FV(reco_nu_vtx_sce_x, reco_nu_vtx_sce_y, reco_nu_vtx_sce_z):
    
    return reco_nu_vtx_sce_x > FV_X_MIN and reco_nu_vtx_sce_x < FV_X_MAX and\
           reco_nu_vtx_sce_y > FV_Y_MIN and reco_nu_vtx_sce_y < FV_Y_MAX and\
           reco_nu_vtx_sce_z > FV_Z_MIN and reco_nu_vtx_sce_z < FV_Z_MAX and\
           not (reco_nu_vtx_sce_z > DEAD_Z_MIN and reco_nu_vtx_sce_z < DEAD_Z_MAX)

################################################################################
# Point located in proton containment volume (PCV) - same as Fiducial volume for now
# Helper function

PCV_X_MIN =   10.0
PCV_X_MAX =  246.4
PCV_Y_MIN = -101.5
PCV_Y_MAX =  101.5
PCV_Z_MIN =   10.0
PCV_Z_MAX = 986.8

def in_proton_containment_vol(x,y,z):
    
    return x > PCV_X_MIN and x < PCV_X_MAX and\
           y > PCV_Y_MIN and y < PCV_Y_MAX and\
           z > PCV_Z_MIN and z < PCV_Z_MAX

################################################################################
# Make vector indicating if tracks/showers are in the containment volume (i.e. checking end is contained)
# Helper function

def is_contained_v(track_end_sce_x_v, track_end_sce_y_v, track_end_sce_z_v):

    contained = []
    for i in range(0,len(track_end_sce_x_v)):
         contained.append(in_proton_containment_vol(track_end_sce_x_v[i], track_end_sce_y_v[i], track_end_sce_z_v[i]))

    return contained

################################################################################
# Check track/shower is contained
# Helper function

def is_contained(Idx, IsContained_v):
    if Idx == -1:
        return False
    return IsContained_v[Idx]

################################################################################
# Returns true if all tracks/showers are contained (i.e. checking generation=2 and start is contained)
# Helper function

def pfp_starts_in_PCV_v(trk_sce_start_x_v, trk_sce_start_y_v, trk_sce_start_z_v):
    
    contained = []
    for i in range(0,len(trk_sce_start_x_v)):
         contained.append(in_proton_containment_vol(trk_sce_start_x_v[i], trk_sce_start_y_v[i], trk_sce_start_z_v[i]))

    return contained

################################################################################
# Returns whether the track/shower passes momentum cuts
# Currently not using upper momentum threshold
# Helper function

proton_p_min = 0.300 #GeV
proton_p_max = 3.0
proton_mass = 0.939
proton_E_min = np.sqrt(proton_p_min**2 + proton_mass**2)
proton_E_max = np.sqrt(proton_p_max**2 + proton_mass**2)

#def pass_mom_cut(RecoMomentum,CUT_LOW,CUT_HIGH):
def pass_mom_cut(RecoMomentum,CUT_LOW):
    if math.isnan(RecoMomentum):
        return False
    
    #return CUT_LOW < RecoMomentum < CUT_HIGH 
    return RecoMomentum > CUT_LOW

################################################################################
# Get indices of reconstructed showers with a starting point within FV

TRACK_SCORE_CUT = 0.5

def reco_showers_v(pfp_generation_v, trk_score_v, pfp_starts_in_PCV_v):
    idx = []
    for i in range(0,len(pfp_generation_v)):
        if pfp_generation_v[i] == 2 and trk_score_v[i] < TRACK_SCORE_CUT and pfp_starts_in_PCV_v[i]:
            idx.append(i)
    
    return idx

################################################################################
# Get number of reconstructed showers with a starting point within FV

def n_reco_showers(reco_showers_v):

    return len(reco_showers_v)

################################################################################
# Get (electron) shower candidate index above threshold

elec_p_min = 0. #GeV
elec_p_max = 1.2 #GeV
elec_mass = 0.511e-3 #GeV
elec_E_min = np.sqrt(elec_p_min**2 + elec_mass**2)
elec_E_max = np.sqrt(elec_p_max**2 + elec_mass**2) # Upper limit currently unused

def reco_elec_candidate_idx(reco_showers_v, shr_energy_cali):

    if len(reco_showers_v) == 1 and shr_energy_cali > elec_E_min:
    #if len(reco_showers_v) == 1 and elec_E_min < shr_energy_cali < elec_E_max:
        return reco_showers_v[0]
    
    return -1

################################################################################
# Find index of the longest track (no pid)

def find_longest_trk_len_idx(pfp_generation_v, trk_len_v):
    
    longest_idx=-1
    longest_len=-1
    for i in range(0,len(pfp_generation_v)):
        if pfp_generation_v[i] == 2 and trk_len_v[i] > longest_len:
            longest_idx = i
            longest_len = trk_len_v[i]

    return longest_idx

# ################################################################################
# # Make a list of the indices of the protons candidates fully contained in the FV

# DEFAULT_PROTON_PID_CUT = 0.02

# def find_proton_candidates(reco_elec_candidate_idx, longest_trk_len_idx, pfp_generation_v, trk_score_v, trk_len_v, trk_llr_pid_score_v, is_contained_v, pfp_starts_in_PCV_v):

#     PID_CUT = 0.015 * trk_len_v[longest_trk_len_idx] + DEFAULT_PROTON_PID_CUT  # for xsec selection as suggested by Elena
#     proton_candidate_idx_v=[]
#     for i in range(0,len(pfp_generation_v)):
#         if pfp_generation_v[i] != 2 or i == reco_elec_candidate_idx or\
#            trk_score_v[i] < TRACK_SCORE_CUT or trk_len_v[i] < 0.0 or\
#            trk_llr_pid_score_v[i] > DEFAULT_PROTON_PID_CUT or\
#            not (is_contained_v[i] and pfp_starts_in_PCV_v[i]): continue
        
#         proton_candidate_idx_v.append(i) 

#     return proton_candidate_idx_v

################################################################################
# Make a list of the indices of the protons candidates fully contained in the FV

DEFAULT_PROTON_PID_CUT = 0.02

def find_proton_candidates(reco_elec_candidate_idx, longest_trk_len_idx, pfp_generation_v, trk_score_v, trk_len_v, trk_llr_pid_score_v, is_contained_v, pfp_starts_in_PCV_v):

    PID_CUT = 0.015 * trk_len_v[longest_trk_len_idx] + DEFAULT_PROTON_PID_CUT  # for xsec selection as suggested by Elena
    proton_candidate_idx_v=[]
    for i in range(0,len(pfp_generation_v)):
        if pfp_generation_v[i] == 2 and i != reco_elec_candidate_idx and\
           trk_score_v[i] > TRACK_SCORE_CUT and trk_len_v[i] >= 0.0 and\
           trk_llr_pid_score_v[i] < DEFAULT_PROTON_PID_CUT and\
           is_contained_v[i] and pfp_starts_in_PCV_v[i]:
           
           proton_candidate_idx_v.append(i) 

    return proton_candidate_idx_v

################################################################################
# Count the number of proton tracks fully contained in the FV

def n_reco_protons(ProtonCandidateIdx_v):
    return len(ProtonCandidateIdx_v)

################################################################################
# Find index of the longest proton track

def find_leading_proton_candidate(ProtonCandidateIdx_v, trk_len_v):
    
    longest_idx=-1
    longest_len=-1
    for i in range(0,len(trk_len_v)):
        if i in ProtonCandidateIdx_v and trk_len_v[i] > longest_len:
            longest_idx = i
            longest_len = trk_len_v[i]

    return longest_idx

################################################################################
# Get momentum of reconstructed leading proton

MASS_PROTON = 0.939 #0.93827

def get_reco_proton_mom(LeadProtonIdx, trk_energy_proton_v):

    if LeadProtonIdx == -1:
        return np.nan
    ke = trk_energy_proton_v[LeadProtonIdx]
    return np.sqrt(ke**2 + (2*MASS_PROTON*ke))

################################################################################
# Get momentum component of the reconstructed leading proton

def get_reco_proton_mom_comp(LeadProtonIdx, trk_energy_proton_v, trk_dir_v):

    if LeadProtonIdx == -1:
        return np.nan

    ke = trk_energy_proton_v[LeadProtonIdx]
    return np.sqrt(ke**2 + (2*MASS_PROTON*ke)) * trk_dir_v[LeadProtonIdx]

################################################################################
# Get total energy of reconstructed leading proton

def get_reco_proton_E(RecoLeadProtonMomentum):

    return np.sqrt(RecoLeadProtonMomentum**2 + MASS_PROTON**2)

################################################################################
# Get kinetic energy of reconstructed leading proton

def get_reco_proton_KE(LeadProtonIdx, trk_energy_proton_v):

    return trk_energy_proton_v[LeadProtonIdx]

################################################################################
# Get momentum of reconstructed protons. Returns a list

def get_reco_proton_mom_v(ProtonCandidateIdx_v, trk_energy_proton_v):

    mom_v = []
    for i in range(0,len(ProtonCandidateIdx_v)):
        ke = trk_energy_proton_v[ProtonCandidateIdx_v[i]]
        mom_v.append(np.sqrt(ke**2 + (2*MASS_PROTON*ke)))

    return mom_v

################################################################################
# Get momentum of reconstructed protons. Returns a list

def get_reco_proton_mom_comp_v(ProtonCandidateIdx_v, trk_energy_proton_v, trk_dir_v):

    mom_v = []
    for i in range(0,len(ProtonCandidateIdx_v)):
        ke = trk_energy_proton_v[ProtonCandidateIdx_v[i]]
        mom_v.append(np.sqrt(ke**2 + (2*MASS_PROTON*ke))*trk_dir_v[ProtonCandidateIdx_v[i]])

    return mom_v

################################################################################
# Get momentum of reconstructed protons. Returns a list

def get_reco_proton_E_v(RecoProtonMomentum_v):

    E_v = []
    for i in range(0,len(RecoProtonMomentum_v)):
        E_v.append(np.sqrt(RecoProtonMomentum_v[i]**2 + MASS_PROTON**2)) 

    return E_v

In [13]:
def apply_selection_1e1p_tki(up,df):
    
#     df["reco_nu_vtx_sce_x"] = up.array("reco_nu_vtx_sce_x")
#     df["reco_nu_vtx_sce_y"] = up.array("reco_nu_vtx_sce_y")
#     df["reco_nu_vtx_sce_z"] = up.array("reco_nu_vtx_sce_z")

    # Load the extra branches needed 
    df["trk_dir_x_v"] = up.array("trk_dir_x_v")
    df["trk_dir_y_v"] = up.array("trk_dir_y_v")
    df["trk_dir_z_v"] = up.array("trk_dir_z_v")
    df["trk_energy_proton_v"] = up.array("trk_energy_proton_v")
    # df['shr_energy_cali'] = up.array('shr_energy_cali')
    # df['shr_energy'] = up.array('shr_energy')
    # df['shr_px'] = up.array('shr_px')
    # df['shr_py'] = up.array('shr_py')
    # df['shr_pz'] = up.array('shr_pz')
    df["pfp_generation_v"] = up.array("pfp_generation_v")
    df["trk_score_v"] = up.array("trk_score_v")
    df["trk_len_v"] = up.array("trk_len_v")
    df["trk_llr_pid_score_v"] = up.array("trk_llr_pid_score_v")
    df["trk_sce_start_x_v"] = up.array("trk_sce_start_x_v")
    df["trk_sce_start_y_v"] = up.array("trk_sce_start_y_v")
    df["trk_sce_start_z_v"] = up.array("trk_sce_start_z_v")
    df["trk_sce_end_x_v"] = up.array("trk_sce_end_x_v")
    df["trk_sce_end_y_v"] = up.array("trk_sce_end_y_v")
    df["trk_sce_end_z_v"] = up.array("trk_sce_end_z_v")

    df["InFV"] = df.apply(lambda x: (sel_reco_vertex_in_FV(x["reco_nu_vtx_sce_x"], x["reco_nu_vtx_sce_y"], x["reco_nu_vtx_sce_z"])), axis=1)
    df["PFPStartsInPCV_v"] = df.apply(lambda x: (pfp_starts_in_PCV_v(x["trk_sce_start_x_v"], x["trk_sce_start_y_v"], x["trk_sce_start_z_v"])), axis=1)
    df["IsContained_v"] = df.apply(lambda x: (is_contained_v(x["trk_sce_end_x_v"], x["trk_sce_end_y_v"], x["trk_sce_end_z_v"])),axis=1)

    # Making corrections to the electron energy and momentum variables
    df['RecoElecMomX'] = df['shr_px'] * df['shr_energy_cali'] / df['shr_energy'] / 0.83
    df['RecoElecMomY'] = df['shr_py'] * df['shr_energy_cali'] / df['shr_energy'] / 0.83
    df['RecoElecMomZ'] = df['shr_pz'] * df['shr_energy_cali'] / df['shr_energy'] / 0.83
    df['RecoElecE'] = df['shr_energy_cali'] * 1/0.83
    df["RecoElecKE"] = df["RecoElecE"] - elec_mass
    df['RecoElecModMom'] = np.sqrt((df['RecoElecMomX'])**2 + (df['RecoElecMomY'])**2 + (df['RecoElecMomZ'])**2)

    df["RecoElecPassMomCut"] = df.apply(lambda x: (pass_mom_cut(x["RecoElecModMom"], elec_p_min)),axis=1) 

    df["RecoShowersIndices"] = df.apply(lambda x: (reco_showers_v(x["pfp_generation_v"], x["trk_score_v"], x["PFPStartsInPCV_v"])), axis=1)
    df["RecoElectronCandidateIdx"] = df.apply(lambda x: (reco_elec_candidate_idx(x["RecoShowersIndices"], x["shr_energy_cali"])), axis=1)
    df["n_reco_showers"] = df.apply(lambda x: (n_reco_showers(x["RecoShowersIndices"])), axis=1)
    df["ElectronFullyContained"] = df.apply(lambda x: (is_contained(x["RecoElectronCandidateIdx"], x["IsContained_v"])),axis=1)

    df["longest_trk_len_idx"] = df.apply(lambda x: (find_longest_trk_len_idx(x["pfp_generation_v"], x["trk_len_v"])), axis=1)
    df["RecoProtonIndices"] = df.apply(lambda x: (find_proton_candidates(x["RecoElectronCandidateIdx"], x["longest_trk_len_idx"], x["pfp_generation_v"], x["trk_score_v"], x["trk_len_v"], x["trk_llr_pid_score_v"], x["IsContained_v"], x["PFPStartsInPCV_v"])), axis=1)
    df["RecoLeadProtonCandidateIdx"] = df.apply(lambda x: (find_leading_proton_candidate(x["RecoProtonIndices"], x["trk_len_v"])), axis=1)
    df["n_reco_tracks"] = df.apply(lambda x: (n_reco_protons(x["RecoProtonIndices"])), axis=1)

    df['RecoLeadProtonModMom'] = df.apply(lambda x: (get_reco_proton_mom(x["RecoLeadProtonCandidateIdx"], x["trk_energy_proton_v"])), axis=1)
    df["RecoLeadProtonMomX"] = df.apply(lambda x: (get_reco_proton_mom_comp(x["RecoLeadProtonCandidateIdx"], x["trk_energy_proton_v"], x["trk_dir_x_v"])), axis=1)
    df["RecoLeadProtonMomY"] = df.apply(lambda x: (get_reco_proton_mom_comp(x["RecoLeadProtonCandidateIdx"], x["trk_energy_proton_v"], x["trk_dir_y_v"])), axis=1)
    df["RecoLeadProtonMomZ"] = df.apply(lambda x: (get_reco_proton_mom_comp(x["RecoLeadProtonCandidateIdx"], x["trk_energy_proton_v"], x["trk_dir_z_v"])), axis=1)
    df["RecoLeadProtonE"] = df.apply(lambda x: (get_reco_proton_E(x["RecoLeadProtonModMom"])),axis=1)
    df["RecoLeadProtonKE"] = df.apply(lambda x: (get_reco_proton_KE(x["RecoLeadProtonCandidateIdx"], x["trk_energy_proton_v"])),axis=1)

    df["RecoLeadProtonPassMomCut"] = df.apply(lambda x: (pass_mom_cut(x["RecoLeadProtonModMom"], proton_p_min)),axis=1)
    
    # Set the reco signal definition
    #nue_cc0piNp = ((df["RecoElectronCandidateIdx"] != -1) & (df["RecoLeadProtonCandidateIdx"] != -1) & (df["InFV"] == True) & (df["RecoElecPassMomCut"] == True) & (df["RecoLeadProtonPassMomCut"] == True))
    nue_cc0pi1p = ((df["RecoElectronCandidateIdx"] != -1) & (df["RecoLeadProtonCandidateIdx"] != -1) & (df["InFV"] == True) & (df["RecoElecPassMomCut"] == True) & (df["RecoLeadProtonPassMomCut"] == True) & (df["n_reco_tracks"] == 1) & (df["n_reco_showers"] == 1))
    
    # df.loc[nue_cc0piNp, "Signal_1eNp"] = True
    # df.loc[~nue_cc0piNp, "Signal_1eNp"] = False
    
    df.loc[nue_cc0pi1p, "Sel_1e1p"] = True
    df.loc[~nue_cc0pi1p, "Sel_1e1p"] = False

    print("Calc reco TKI variables for leading proton only")

    df["RecoDeltaPT"] = df.apply(lambda x: (tki_calculators.delta_pT(x["RecoElecMomX"],x["RecoElecMomY"],x["RecoElecMomZ"],x["RecoLeadProtonMomX"],x["RecoLeadProtonMomY"],x["RecoLeadProtonMomZ"])),axis=1)
    #df["RecoDeltaPhiT"] = df.apply(lambda x: (tki_calculators.delta_phiT(x["RecoElecMomX"],x["RecoElecMomY"],x["RecoElecMomZ"],x["RecoLeadProtonMomX"],x["RecoLeadProtonMomY"],x["RecoLeadProtonMomZ"])),axis=1)
    df["RecoDeltaAlphaT"] = df.apply(lambda x: (tki_calculators.delta_alphaT(x["RecoElecMomX"],x["RecoElecMomY"],x["RecoElecMomZ"],x["RecoLeadProtonMomX"],x["RecoLeadProtonMomY"],x["RecoLeadProtonMomZ"])),axis=1)
    df['RecoDeltaAlphaT'] = np.degrees(df['RecoDeltaAlphaT'])

    # Drop all of the temporary columns added to the dataframe to save space
    df.drop("pfp_generation_v",inplace=True,axis=1)
    df.drop("trk_score_v",inplace=True,axis=1)
    df.drop("trk_len_v",inplace=True,axis=1)
    df.drop("trk_llr_pid_score_v",inplace=True,axis=1)
    df.drop("trk_sce_start_x_v",inplace=True,axis=1)
    df.drop("trk_sce_start_y_v",inplace=True,axis=1)
    df.drop("trk_sce_start_z_v",inplace=True,axis=1)
    df.drop("trk_sce_end_x_v",inplace=True,axis=1)
    df.drop("trk_sce_end_y_v",inplace=True,axis=1)
    df.drop("trk_sce_end_z_v",inplace=True,axis=1)
    df.drop("trk_energy_proton_v",inplace=True,axis=1)
    df.drop("trk_dir_x_v",inplace=True,axis=1)
    df.drop("trk_dir_y_v",inplace=True,axis=1)
    df.drop("trk_dir_z_v",inplace=True,axis=1)

    return df

In [14]:
# Functions for setting the signal definition in truth variables and adding useful variables for the CC1e1P selection
# (original framework in Root/C++ developed by S Gardiner, re-written into the python PeLEE framework by C Thorpe)
# Author: M Moudgalya

################################################################################
# Check there is a final state muon above threshold, and return its index

muon_p_min = 0.1
muon_p_max = 1.2
muon_mass = 0.1057
muon_E_min = np.sqrt(muon_p_min**2 + muon_mass**2)
muon_E_max = np.sqrt(muon_p_max**2 + muon_mass**2) # SG's code imposes an upper limit

def true_muon_idx(mc_pdg,mc_E):

    for i in range(0,len(mc_pdg)):
        if abs(mc_pdg[i]) == 13 and muon_E_min < mc_E[i] < muon_E_max:
            return i

    return -1

################################################################################
# Check there is a final state electron above threshold, and return its index

elec_p_min = 0. #GeV
elec_p_max = 1.2 #GeV
elec_mass = 0.511e-3 #GeV
elec_E_min = np.sqrt(elec_p_min**2 + elec_mass**2)
elec_E_max = np.sqrt(elec_p_max**2 + elec_mass**2) # Upper limit currently unused

def true_elec_idx(mc_pdg,mc_E):

    for i in range(0,len(mc_pdg)):
        if abs(mc_pdg[i]) == 11 and mc_E[i] > elec_E_min:
        #if abs(mc_pdg[i]) == 11 and elec_E_min < mc_E[i] < elec_E_max:
            return i

    return -1

################################################################################
# Return indices of electrons above threshold

def true_elec_indices(mc_pdg,mc_E):

    idx = []
    for i in range(0,len(mc_pdg)):
        if abs(mc_pdg[i]) == 11 and mc_E[i] > elec_E_min:
        #if abs(mc_pdg[i]) == 11 and elec_E_min < mc_E[i] < elec_E_max:
            idx.append(i)

    return idx

################################################################################
# Number of electrons above threshold

def n_elec(TrueIdx_v):

    return len(TrueIdx_v) 

################################################################################
# Final state has one proton above threshold

proton_p_min = 0.300 #GeV
proton_p_max = 3.0
proton_mass = 0.939
proton_E_min = np.sqrt(proton_p_min**2 + proton_mass**2)
proton_E_max = np.sqrt(proton_p_max**2 + proton_mass**2) # Upper limit currently unused

def true_proton_idx(mc_pdg,mc_E):

    idx = []
    for i in range(0,len(mc_pdg)):
        if abs(mc_pdg[i]) == 2212 and mc_E[i] > proton_E_min:
        #if abs(mc_pdg[i]) == 2212 and proton_E_min < mc_E[i] < proton_E_max:
            idx.append(i)

    return idx

################################################################################
# Number of protons above threshold

def n_proton(TrueIdx_v):

    return len(TrueIdx_v) 

################################################################################
# Index of lead proton 

def true_lead_proton_idx(mc_pdg,mc_E):

    lead_idx = -1
    lead_E = -1
    for i in range(0,len(mc_pdg)):
        if abs(mc_pdg[i]) == 2212 and mc_E[i] > proton_E_min and mc_E[i] > lead_E:
        #if abs(mc_pdg[i]) == 2212 and proton_E_min < mc_E[i] < proton_E_max and mc_E[i] > lead_E:
            lead_E = mc_E[i]
            lead_idx = i

    return lead_idx

################################################################################
# Final state has no charged pions above thresold and no pi0

pion_thresh = 0.07 #GeV
pion_mass = 0.1396
pion_E_thresh = np.sqrt(pion_thresh**2 + pion_mass**2)

def has_no_mesons(mc_pdg,mc_E):
    for i in range(0,len(mc_pdg)):
        if (abs(mc_pdg[i]) == 211 and mc_E[i] > pion_E_thresh) or mc_pdg[i] == 111 or abs(mc_pdg[i]) == 321:
            return False 

    return True

################################################################################
# Number of charged pions above threshold

def n_fs_pion(mc_pdg,mc_E):

    n_pion = 0
    for i in range(0,len(mc_pdg)):
        if abs(mc_pdg[i]) == 211 and mc_E[i] > pion_E_thresh:
            n_pion += 1

    return n_pion

################################################################################
# Number of neutral pions

def n_fs_pi0(mc_pdg,mc_E):

    n_pi0 = 0
    for i in range(0,len(mc_pdg)):
        if mc_pdg[i] == 111:
            n_pi0 += 1

    return n_pi0

################################################################################
# True primary vertex is in the fiducial volume

# Definitions in PeLEE technote and 'selected' variable
# https://github.com/ubneutrinos/searchingfornues/blob/0489ac5457335a553a3bab54ee5d7ba91734adf0/Selection/SelectionTools/CC0piNpSelection_tool.cc#L97-L102
# https://github.com/ubneutrinos/searchingfornues/blob/889002e5ec93b567265c3af8c178172363200490/Selection/SelectionTools/CC0piNpSelection_tool.cc#L415-L420
FV_X_MIN =   10.0
FV_X_MAX =  246.4
FV_Y_MIN = -101.5
FV_Y_MAX =  101.5
FV_Z_MIN =   10.0
FV_Z_MAX =  986.8
DEAD_Z_MIN = 675 # SG's code does not cut dead region 
DEAD_Z_MAX = 775

def in_fiducial_volume(true_nu_vtx_x,true_nu_vtx_y,true_nu_vtx_z):

    return FV_X_MIN < true_nu_vtx_x < FV_X_MAX and\
           FV_Y_MIN < true_nu_vtx_y < FV_Y_MAX and\
           FV_Z_MIN < true_nu_vtx_z < FV_Z_MAX and\
           not (DEAD_Z_MIN < true_nu_vtx_z < DEAD_Z_MAX)

################################################################################
# Component of the momentum of a particle at index TrueIdx

def true_mom(TrueIdx,mc_p):

    if TrueIdx == -1:
        return np.nan
    else:
        return mc_p[TrueIdx]
    
################################################################################
# Component of the total momentum of a group of particles 

def true_mom_tot(TrueIdx_v,mc_p):

    if len(TrueIdx_v) == 0:
        return np.nan

    p = 0
    for i in TrueIdx_v:
        p = p + mc_p[i]    

    return p 

################################################################################
# Vector of momenta of group of particles 

def true_mom_v(TrueIdx_v,mc_p):

    #p = np.array([])
    p = []
    for i in TrueIdx_v:
        #p = np.append(p, mc_p[i])
        p.append(mc_p[i])

    return p 

In [15]:
def set_Signal1e1p(up,df):

    # Load the extra branches needed 
    df["mc_pdg"] = up.array("mc_pdg")
    df["mc_E"] = up.array("mc_E")
    df["mc_px"] = up.array("mc_px")
    df["mc_py"] = up.array("mc_py")
    df["mc_pz"] = up.array("mc_pz")

    df["InFV"] = df.apply(lambda x: (in_fiducial_volume(x["true_nu_vtx_x"],x["true_nu_vtx_y"],x["true_nu_vtx_z"])),axis=1)
    
    df["TrueMuonIdx"] = df.apply(lambda x: (true_muon_idx(x["mc_pdg"],x["mc_E"])),axis=1)
    df["TrueElecIdx"] = df.apply(lambda x: (true_elec_idx(x["mc_pdg"],x["mc_E"])),axis=1)
    df["TrueLeadProtonIdx"] = df.apply(lambda x: (true_lead_proton_idx(x["mc_pdg"],x["mc_E"])),axis=1)
    
    df["TrueProtonIdx"] = df.apply(lambda x: (true_proton_idx(x["mc_pdg"],x["mc_E"])),axis=1)
    df["TrueNProt"] = df.apply(lambda x: (n_proton(x["TrueProtonIdx"])),axis=1)
    df["TrueFSPions"] = df.apply(lambda x: (n_fs_pion(x["mc_pdg"],x["mc_E"])),axis=1)
    df["TrueFSPi0"] = df.apply(lambda x: (n_fs_pi0(x["mc_pdg"],x["mc_E"])),axis=1)
    df["HasNoMesons"] = df.apply(lambda x: (has_no_mesons(x["mc_pdg"],x["mc_E"])),axis=1)
    df["TrueElecIndices"] = df.apply(lambda x: (true_elec_indices(x["mc_pdg"],x["mc_E"])),axis=1)
    df["TrueNElec"] = df.apply(lambda x: (n_elec(x["TrueElecIndices"])),axis=1)

    df["TrueElecE"] = df.apply(lambda x: (true_mom(x["TrueElecIdx"],x["mc_E"])),axis=1)
    df["TrueElecMomX"] = df.apply(lambda x: (true_mom(x["TrueElecIdx"],x["mc_px"])),axis=1)
    df["TrueElecMomY"] = df.apply(lambda x: (true_mom(x["TrueElecIdx"],x["mc_py"])),axis=1)
    df["TrueElecMomZ"] = df.apply(lambda x: (true_mom(x["TrueElecIdx"],x["mc_pz"])),axis=1)

    df["TrueElecKE"] = df["TrueElecE"] - elec_mass
    df['TrueElecModMom'] = np.sqrt((df['TrueElecMomX'])**2 + (df['TrueElecMomY'])**2 + (df['TrueElecMomZ'])**2)

    df["TrueLeadProtonE"] = df.apply(lambda x: (true_mom(x["TrueLeadProtonIdx"],x["mc_E"])),axis=1)
    df["TrueLeadProtonMomX"] = df.apply(lambda x: (true_mom(x["TrueLeadProtonIdx"],x["mc_px"])),axis=1)
    df["TrueLeadProtonMomY"] = df.apply(lambda x: (true_mom(x["TrueLeadProtonIdx"],x["mc_py"])),axis=1)
    df["TrueLeadProtonMomZ"] = df.apply(lambda x: (true_mom(x["TrueLeadProtonIdx"],x["mc_pz"])),axis=1)

    df["TrueLeadProtonKE"] = df["TrueLeadProtonE"] - proton_mass
    df['TrueLeadProtonModMom'] = np.sqrt((df['TrueLeadProtonMomX'])**2 + (df['TrueLeadProtonMomY'])**2 + (df['TrueLeadProtonMomZ'])**2)
    
    # Set the signal definition
    nue_cc0piNp = ((abs(df["nu_pdg"]) == 12) & (df["TrueElecIdx"] != -1) & (df["TrueLeadProtonIdx"] != -1) & (df["InFV"] == True) & (df["HasNoMesons"] == True))
    nue_cc0pi1p = ((abs(df["nu_pdg"]) == 12) & (df["TrueElecIdx"] != -1) & (df["TrueLeadProtonIdx"] != -1) & (df["InFV"] == True) & (df["HasNoMesons"] == True) & (df["TrueNProt"] == 1))
    
    df.loc[nue_cc0piNp, "Signal_1eNp"] = True
    df.loc[~nue_cc0piNp, "Signal_1eNp"] = False
    
    df.loc[nue_cc0pi1p, "Signal_1e1p"] = True
    df.loc[~nue_cc0pi1p, "Signal_1e1p"] = False
    
    # Set the topological categories
    
    nue_cc0pi0p = ((abs(df["nu_pdg"]) == 12) & (df["TrueElecIdx"] != -1) & (df["TrueLeadProtonIdx"] == -1) & (df["InFV"] == True) & (df["HasNoMesons"] == True) & (df["TrueNProt"] == 0))
    nue_cc0pi2p = ((abs(df["nu_pdg"]) == 12) & (df["TrueElecIdx"] != -1) & (df["TrueLeadProtonIdx"] != -1) & (df["InFV"] == True) & (df["HasNoMesons"] == True) & (df["TrueNProt"] >= 2))
    nue_cc = ((abs(df["nu_pdg"]) == 12) & (df["TrueElecIdx"] != -1) & (df["InFV"] == True) & (df["TrueFSPions"] > 0) & (df["TrueFSPi0"] > 0) & (df["TrueNProt"] >= 0))
    nue_nc0pi0 = ((abs(df["nu_pdg"]) == 12) & (df["TrueElecIdx"] == -1) & (df["InFV"] == True) & (df["TrueFSPions"] == 0) & (df["TrueFSPi0"] == 0) & (df["TrueNProt"] == 0))
    nue_ncNpi0 = ((abs(df["nu_pdg"]) == 12) & (df["TrueElecIdx"] == -1) & (df["InFV"] == True) & (df["TrueFSPions"] == 0) & (df["TrueFSPi0"] >= 1) & (df["TrueNProt"] == 0))
    
    numu_ccNpi0 = ((abs(df["nu_pdg"]) == 14) & (df["TrueMuonIdx"] != -1) & (df["InFV"] == True) & (df["TrueFSPions"] >= 0) & (df["TrueFSPi0"] >= 1) & (df["TrueNProt"] >= 0))
    numu_cc0pi0 = ((abs(df["nu_pdg"]) == 14) & (df["TrueMuonIdx"] != -1) & (df["InFV"] == True) & (df["TrueFSPions"] >= 0) & (df["TrueFSPi0"] == 1) & (df["TrueNProt"] >= 0))
    numu_nc0pi0 = ((abs(df["nu_pdg"]) == 14) & (df["TrueMuonIdx"] == -1) & (df["InFV"] == True) & (df["TrueFSPions"] == 0) & (df["TrueFSPi0"] == 0) & (df["TrueNProt"] == 0))
    numu_ncNpi0 = ((abs(df["nu_pdg"]) == 14) & (df["TrueMuonIdx"] == -1) & (df["InFV"] == True) & (df["TrueFSPions"] == 0) & (df["TrueFSPi0"] >= 1) & (df["TrueNProt"] == 0))
    
    outFV = df["InFV"] == False
    # cosmic = ((abs(df["nu_pdg"]) != 14) & (df["nu_pdg"] != 12) & (df["InFV"] == True)) #logic used in PeLEE analyser (DefaultAnalysis_tool.cc)
    cosmic = ((abs(df["nu_pdg"]) != 14) & (df["nu_pdg"] != 12) & (df["InFV"] == True) & ~nue_cc0piNp & ~nue_cc0pi1p & ~nue_cc0pi0p & ~nue_cc0pi2p & ~nue_cc & ~nue_nc0pi0 & ~nue_ncNpi0 & ~numu_ccNpi0 & ~numu_cc0pi0 & ~numu_nc0pi0 & ~numu_ncNpi0)

    # cosmic_old = ((df["InFV"] == True) & ~nue_cc0piNp & ~nue_cc0pi1p & ~nue_cc0pi0p & ~nue_cc0pi2p & ~nue_cc & ~nue_nc0pi0 & ~nue_ncNpi0 & ~numu_ccNpi0 & ~numu_cc0pi0 & ~numu_nc0pi0 & ~numu_ncNpi0)
    # cosmic_new = ((df["InFV"] == True) & (df["category"] == 4) & ~nue_cc0piNp & ~nue_cc0pi1p & ~nue_cc0pi0p & ~nue_cc0pi2p & ~nue_cc & ~nue_nc0pi0 & ~nue_ncNpi0 & ~numu_ccNpi0 & ~numu_cc0pi0 & ~numu_nc0pi0 & ~numu_ncNpi0)
    # cosmic_newest = ((df["InFV"] == True) & (df["category"] == 4))

    # df["test"] = 6
    # df.loc[nue_cc0pi0p, "test"] = 10
    # df.loc[nue_cc0pi1p, "test"] = 12
    # df.loc[nue_cc0pi2p, "test"] = 13
    # df.loc[nue_cc, "test"] = 1
    # df.loc[nue_nc0pi0, "test"] = 3
    # df.loc[nue_ncNpi0, "test"] = 31
    # df.loc[numu_cc0pi0, "test"] = 2
    # df.loc[numu_ccNpi0, "test"] = 21
    # df.loc[numu_nc0pi0, "test"] = 3
    # df.loc[numu_ncNpi0, "test"] = 31
    # df.loc[outFV, "test"] = 5
    # df.loc[cosmic_old, "test"] = 4

    # df["test2"] = 6
    # df.loc[nue_cc0pi0p, "test2"] = 10
    # df.loc[nue_cc0pi1p, "test2"] = 12
    # df.loc[nue_cc0pi2p, "test2"] = 13
    # df.loc[nue_cc, "test2"] = 1
    # df.loc[nue_nc0pi0, "test2"] = 3
    # df.loc[nue_ncNpi0, "test2"] = 31
    # df.loc[numu_cc0pi0, "test2"] = 2
    # df.loc[numu_ccNpi0, "test2"] = 21
    # df.loc[numu_nc0pi0, "test2"] = 3
    # df.loc[numu_ncNpi0, "test2"] = 31
    # df.loc[outFV, "test2"] = 5
    # df.loc[cosmic_new, "test2"] = 4

    # df["test3"] = 6
    # df.loc[cosmic, "test3"] = 4
    # df.loc[nue_cc0pi0p, "test3"] = 10
    # df.loc[nue_cc0pi1p, "test3"] = 12
    # df.loc[nue_cc0pi2p, "test3"] = 13
    # df.loc[nue_cc, "test3"] = 1
    # df.loc[nue_nc0pi0, "test3"] = 3
    # df.loc[nue_ncNpi0, "test3"] = 31
    # df.loc[numu_cc0pi0, "test3"] = 2
    # df.loc[numu_ccNpi0, "test3"] = 21
    # df.loc[numu_nc0pi0, "test3"] = 3
    # df.loc[numu_ncNpi0, "test3"] = 31
    # df.loc[outFV, "test3"] = 5
    # #df.loc[cosmic, "test3"] = 4

    # df["test4"] = 6
    # df.loc[nue_cc0pi0p, "test4"] = 10
    # df.loc[nue_cc0pi1p, "test4"] = 12
    # df.loc[nue_cc0pi2p, "test4"] = 13
    # df.loc[nue_cc, "test4"] = 1
    # df.loc[nue_nc0pi0, "test4"] = 3
    # df.loc[nue_ncNpi0, "test4"] = 31
    # df.loc[numu_cc0pi0, "test4"] = 2
    # df.loc[numu_ccNpi0, "test4"] = 21
    # df.loc[numu_nc0pi0, "test4"] = 3
    # df.loc[numu_ncNpi0, "test4"] = 31
    # df.loc[outFV, "test4"] = 5
    # df.loc[cosmic_not, "test4"] = 4

    df["category_1e1p_tki"] = 6  # 'other'
    #df.loc[cosmic_newest, "category_1e1p_tki"] = 4
    df.loc[nue_cc0pi0p, "category_1e1p_tki"] = 10
    df.loc[nue_cc0pi1p, "category_1e1p_tki"] = 12  # 1e1p signal
    df.loc[nue_cc0pi2p, "category_1e1p_tki"] = 13
    df.loc[nue_cc, "category_1e1p_tki"] = 1
    df.loc[nue_nc0pi0, "category_1e1p_tki"] = 3
    df.loc[nue_ncNpi0, "category_1e1p_tki"] = 31
    df.loc[numu_cc0pi0, "category_1e1p_tki"] = 2
    df.loc[numu_ccNpi0, "category_1e1p_tki"] = 21
    df.loc[numu_nc0pi0, "category_1e1p_tki"] = 3
    df.loc[numu_ncNpi0, "category_1e1p_tki"] = 31
    df.loc[outFV, "category_1e1p_tki"] = 5
    df.loc[cosmic, "category_1e1p_tki"] = 4

    # Calculate the TKI variables for 1e1p using the leading proton

    print("Calc true TKI variables for leading proton only")

    df["TrueDeltaPT"] = df.apply(lambda x: (tki_calculators.delta_pT(x["TrueElecMomX"],x["TrueElecMomY"],x["TrueElecMomZ"],x["TrueLeadProtonMomX"],x["TrueLeadProtonMomY"],x["TrueLeadProtonMomZ"])),axis=1)
    #df["TrueDeltaPhiT"] = df.apply(lambda x: (tki_calculators.delta_phiT(x["TrueElecMomX"],x["TrueElecMomY"],x["TrueElecMomZ"],x["TrueLeadProtonMomX"],x["TrueLeadProtonMomY"],x["TrueLeadProtonMomZ"])),axis=1)
    df["TrueDeltaAlphaT"] = df.apply(lambda x: (tki_calculators.delta_alphaT(x["TrueElecMomX"],x["TrueElecMomY"],x["TrueElecMomZ"],x["TrueLeadProtonMomX"],x["TrueLeadProtonMomY"],x["TrueLeadProtonMomZ"])),axis=1)
    df['TrueDeltaAlphaT'] = np.degrees(df['TrueDeltaAlphaT'])
    
    
    # Drop temporary data from dataframes
    #df.drop("mc_pdg", inplace=True, axis=1)
    df.drop("mc_E", inplace=True, axis=1)
    df.drop("mc_px", inplace=True, axis=1)
    df.drop("mc_py", inplace=True, axis=1)
    df.drop("mc_pz", inplace=True, axis=1)
    
    return df

In [16]:
def add_paper_category_1e1p(df):
    df.loc[:, "category_1e1p"] = df["category"]
    nue_cc0pi1p = ((abs(df["nu_pdg"]) == 12) & (df["TrueElecIdx"] != -1) & (df["TrueLeadProtonIdx"] != -1) & (df["InFV"] == True) & (df["HasNoMesons"] == True) & (df["TrueNElec"] == 1) & (df["TrueNProt"] == 1))
    nue_cc0pi2p = ((abs(df["nu_pdg"]) == 12) & (df["TrueElecIdx"] != -1) & (df["TrueLeadProtonIdx"] != -1) & (df["InFV"] == True) & (df["HasNoMesons"] == True) & (df["TrueNElec"] == 1) & (df["TrueNProt"] >= 2))
    df.loc[df["category"].isin([11]) & nue_cc0pi1p, "category_1e1p"] = 12
    df.loc[df["category"].isin([11]) & nue_cc0pi2p, "category_1e1p"] = 13
    return df

In [17]:
import uproot
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

fold = "nuselection"
tree = "NeutrinoSelectionFilter"
path = "/uboone/data/users/cthorpe/PELEE_2023_Samples/"
RUN1 = "run1/nuepresel/"
RUN2 = "run2/nuepresel/"
RUN3 = "run3/nuepresel/"
RUN4 = "run4/nuepresel/"
RUN5 = "run5/nuepresel/"
fn = "overlay_peleeTuple_uboone_v08_00_00_70_run3_nu.root"

data_path = path + RUN3 + fn

#df = uproot.open('/pnfs/uboone/persistent/users/jdetje/pelee_v08_00_00_70/overlay_peleeTuple_uboone_v08_00_00_70_run3_nu.root')[fold][tree].pandas.df(flatten=False)
variables = ['category', 'nu_pdg', 'true_nu_vtx_x', 'true_nu_vtx_y', 'true_nu_vtx_z', 
             'reco_nu_vtx_sce_x', 'reco_nu_vtx_sce_y', 'reco_nu_vtx_sce_z', 
             'shr_energy_cali', 'shr_energy', 'shr_px', 'shr_py', 'shr_pz', 
             'nslice', 'selected', 'shr_energy_tot_cali', '_opfilter_pe_beam', '_opfilter_pe_veto', 
            'CosmicIPAll3D', 'hits_ratio', 'shrmoliereavg', 'subcluster', 'trkfit', 'tksh_distance', 
            'shr_tkfit_dedx_max', 'tksh_angle']

up = uproot.open('/exp/uboone/data/users/cthorpe/PELEE_2023_Samples/run3/nuepresel/overlay_peleeTuple_uboone_v08_00_00_70_run3_nu.root')[fold][tree]
df = up.pandas.df(variables, flatten=False)
#df = up.pandas.df(flatten=False)

In [18]:
run3mc = apply_selection_1e1p_tki(up,df)
run3mc = set_Signal1e1p(up,run3mc)
run3mc = add_paper_category_1e1p(run3mc)
print(type(run3mc))

Calc reco TKI variables for leading proton only
Calc true TKI variables for leading proton only
<class 'pandas.core.frame.DataFrame'>


In [14]:
print("Signal_1e1p" in run3mc.columns)
run3mc.loc[:, ('Signal_1e1p','Sel_1e1p')].head(10)

True


,Signal_1e1p,Sel_1e1p
entry,,
0,False,False
1,False,False
2,False,False
3,False,False
4,False,True
5,False,False
6,True,True
7,False,False
8,False,False


In [23]:
is_sig = run3mc['Signal_1e1p'] == True
all_sig = run3mc.loc[is_sig]

top_sig = run3mc['category_1e1p'] == 12
all_top_sig = run3mc.loc[top_sig]

test_sig = run3mc['Sel_1e1p'] == True
all_test_sig = run3mc.loc[test_sig]

print('Signal_1e1p:', len(all_sig))
print('category_1e1p:', len(all_top_sig))
print('Sel_1e1p:', len(all_test_sig))
print(len(run3mc))

Signal_1e1p: 322
category_1e1p: 322
Sel_1e1p: 4161
32186


In [20]:
from microfit import selections as sel

#print(sel.selection_categories)

# selection = "OnePL_new"
# preselection = "OneP_new"
# query = f"{sel.preselection_categories[preselection]['query']} and {sel.selection_categories[selection]['query']}"

# nue preselection
PRESQ = 'nslice == 1'
PRESQ += ' and selected == 1'
PRESQ += ' and shr_energy_tot_cali > 0.07'
PRESQ += ' and ( (_opfilter_pe_beam > 0 and _opfilter_pe_veto < 20))'

# 1e1p preselection - new
OnePPRESQ_new = PRESQ
OnePPRESQ_new += ' and Sel_1e1p == True'

# 1e1p selection (loose box cuts, same as 1eNp loose box cuts)
OnePLCUTQ_new = OnePPRESQ_new
OnePLCUTQ_new += ' and CosmicIPAll3D > 10.'
OnePLCUTQ_new += ' and hits_ratio > 0.50'
OnePLCUTQ_new += ' and shrmoliereavg < 9'
OnePLCUTQ_new += ' and subcluster > 4'
OnePLCUTQ_new += ' and trkfit < 0.65'
OnePLCUTQ_new += ' and tksh_distance < 6.0'
OnePLCUTQ_new += ' and (shr_tkfit_dedx_max > 0.5 and shr_tkfit_dedx_max < 5.5)' 
OnePLCUTQ_new += ' and tksh_angle > -0.9'

query = OnePLCUTQ_new

temp_df = run3mc.query(query, engine='python')
temp_df.head()

,category,nu_pdg,true_nu_vtx_x,true_nu_vtx_y,true_nu_vtx_z,reco_nu_vtx_sce_x,reco_nu_vtx_sce_y,reco_nu_vtx_sce_z,shr_energy_cali,shr_energy,...,TrueLeadProtonMomY,TrueLeadProtonMomZ,TrueLeadProtonKE,TrueLeadProtonModMom,Signal_1eNp,Signal_1e1p,category_1e1p_tki,TrueDeltaPT,TrueDeltaAlphaT,category_1e1p
entry,,,,,,,,,,,,,,,,,,,,,
4,31,14,181.504242,-18.688145,336.550934,182.639847,-18.467836,336.274872,0.161168,0.166402,...,-0.492534,0.208594,0.157898,0.568182,False,False,6,NaN,NaN,31
6,11,12,41.675133,-93.997543,412.291870,42.748440,-93.905602,412.158356,0.730024,0.714216,...,0.232348,0.222041,0.112096,0.473759,True,True,12,0.082011,144.03178,12
355,21,14,168.602585,-29.603479,289.600250,169.701889,-29.619741,289.695740,0.654421,0.653946,...,0.195401,0.801125,0.350237,0.884182,False,False,21,NaN,NaN,21
601,2,14,108.253525,-41.947155,433.742157,109.342819,-41.462643,434.060547,0.393329,0.400269,...,0.324988,0.734701,0.318094,0.836618,False,False,6,NaN,NaN,2
679,11,12,81.071777,46.890144,834.647095,82.378410,46.956028,834.608643,0.818319,0.675749,...,0.018988,0.232800,0.140297,0.533412,True,False,13,0.145942,145.48925,13


In [29]:
reco_sig = temp_df.index
run3mc['new_col'] = False
run3mc.loc[reco_sig, 'new_col'] = True
run3mc.loc[:, ('Signal_1e1p','Sel_1e1p', 'new_col')].head(10)

,Signal_1e1p,Sel_1e1p,new_col
entry,,,
0,False,False,False
1,False,False,False
2,False,False,False
3,False,False,False
4,False,True,True
5,False,False,False
6,True,True,True
7,False,False,False
8,False,False,False


In [15]:
all_mc = pd.concat([df for k, df in rundata.items() if k!='data' or k!='ext'])

In [ ]:
from microfit import selections as sel

#print(sel.selection_categories)

selection = "OnePL"
preselection = "OneP"
query = f"{sel.preselection_categories[preselection]['query']} and {sel.selection_categories[selection]['query']}"
#print(query)

all_mc = pd.concat([df for k, df in rundata.items() if k!='data'])
#is_sig = all_mc['category_1e1p_tki'] == 12
is_sig = all_mc['Signal_1e1p'] == True
all_sig = all_mc.loc[is_sig]
sel_sig = all_sig.query(query, engine='python')

print(len(all_sig))
print()

all_mc.loc[all_mc['category_1e1p_tki'] == 6].head()
print(len(all_mc))
print(len(all_mc.loc[all_mc['category_1e1p_tki'] == 6]))
plt.hist(all_mc['category_1e1p_tki'], 31)